<a href="https://colab.research.google.com/github/fataa34/applied-optimization-with-python/blob/constraint/Constraint_(Indirect).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Constraint - Indirect

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [ ]:
def objective_function(x):
  return x[0]**4-2*x[0]**2*x[1]+x[0]**2+x[0]*x[1]**2-2*x[0]+4

def eq1(x):
  return x[0]**2+x[1]**2-2

def ineq1(x):
  return 0.25*x[0]**2+0.75*x[1]**2-1

## Exterior Penalty Function

In [ ]:
def exterior_penalty_method(target_func, x0, eq_constraints, ineq_constraints,Ns, rh_vec, rg_vec, Ch, Cg, epsilon):

    if len(rh_vec) != len(eq_constraints):
        raise ValueError("Panjang rh_vec harus sama dengan jumlah eq_constraints")
    if len(rg_vec) != len(ineq_constraints):
        raise ValueError("Panjang rg_vec harus sama dengan jumlah ineq_constraints")

    rh_arr = np.array(rh_vec, dtype=float)
    rg_arr = np.array(rg_vec, dtype=float)

    x_current = np.array(x0, dtype=float)
    f_prev = target_func(x_current)

    val_eq_init = [h(x_current) for h in eq_constraints]
    val_ineq_init = [g(x_current) for g in ineq_constraints]
    history_data = []
    history_data.append([0, x_current, f_prev, val_eq_init, val_ineq_init])
    for q in range(1, Ns + 1):
        def pseudo_objective(x):
            penalty = 0
            for r_val, h_func in zip(rh_arr, eq_constraints):
                penalty += r_val * (h_func(x)**2)
            for r_val, g_func in zip(rg_arr, ineq_constraints):
                penalty += r_val * (max(0, g_func(x))**2)
            return target_func(x) + penalty

        res = minimize(pseudo_objective, x_current, method='BFGS')
        x_next = res.x
        f_current = pseudo_objective(x_next)

        max_violation = 0
        current_eq_vals = []
        current_ineq_vals = []

        for h in eq_constraints:
            val = h(x_next)
            current_eq_vals.append(val)
            max_violation = max(max_violation, abs(val))

        for g in ineq_constraints:
            val = g(x_next)
            current_ineq_vals.append(val)
            max_violation = max(max_violation, val)

        delta_F = f_current - f_prev
        delta_X = x_next - x_current
        norm_delta_X_sq = np.dot(delta_X, delta_X)

        stop_cond_1 = (delta_F**2) <= epsilon
        stop_cond_2 = norm_delta_X_sq <= epsilon

        if max_violation <= epsilon and (stop_cond_1 or stop_cond_2):
          history_data.append([q, x_next, f_current, current_eq_vals, current_ineq_vals])
          break
        rh_arr = rh_arr * Ch
        rg_arr = rg_arr * Cg
        f_prev = f_current
        x_current = x_next
        history_data.append([q, x_next, f_current, current_eq_vals, current_ineq_vals])
    return pd.DataFrame(history_data, columns=['iterasi', 'x', 'f(x)', 'h(x)', 'g(x)'])

In [ ]:
rh_init = []
rg_init = [1.0]
x0 = [2,2]
eq_constraints=[]
ineq_constraints=[ineq1]
rh_vec=rh_init
rg_vec=rg_init
Ch=3.0
Cg=0.2
Ns=20
epsilon = 1e-6

exterior_penalty_method(objective_function, x0, eq_constraints, ineq_constraints,Ns, rh_vec, rg_vec, Ch, Cg, epsilon)

,iterasi,x,f(x),h(x),g(x)
0,0,"[2.0, 2.0]",12.00000,[],[3.0]
1,1,"[0.8519702971698252, 0.8519694551026314]",2.93037,[],[-0.27414768886418395]
2,2,"[0.8519702971698252, 0.8519694551026314]",2.93037,[],[-0.27414768886418395]


## Augmented Lagrange

In [ ]:
def augmented_lagrange_method(target_func, x0, eq_constraints, ineq_constraints, Ns, rh_vec, rg_vec, Ch, Cg, r_max, epsilon):
    rh = np.array(rh_vec, dtype=float)
    rg = np.array(rg_vec, dtype=float)

    lam = np.zeros(len(eq_constraints))
    beta = np.zeros(len(ineq_constraints))
    x_current = np.array(x0, dtype=float)
    f_prev = target_func(x_current)
    history_data = []
    val_eq_init = [h(x_current) for h in eq_constraints]
    val_ineq_init = [g(x_current) for g in ineq_constraints]
    history_data.append([0, x_current.copy(), target_func(x_current), val_eq_init, val_ineq_init, np.zeros_like(lam), np.zeros_like(beta)])
    for q in range(1, Ns + 1):

        def alm_objective(x):
            obj_val = target_func(x)

            eq_term = 0
            for i, h_func in enumerate(eq_constraints):
                h_val = h_func(x)
                eq_term += lam[i] * h_val + rh[i] * (h_val**2)

            ineq_term = 0
            for j, g_func in enumerate(ineq_constraints):
                g_val = g_func(x)
                # Syarat aktif: beta + 2*r*g > 0
                term_bracket = max(0, beta[j] + 2 * rg[j] * g_val)
                ineq_term += (1.0 / (4.0 * rg[j])) * (term_bracket**2 - beta[j]**2)

            return obj_val + eq_term + ineq_term

        res = minimize(alm_objective, x_current, method='BFGS')
        x_next = res.x
        f_current_alm = alm_objective(x_next)

        max_violation = 0
        current_eq_vals = []
        current_ineq_vals = []

        for h in eq_constraints:
            val = h(x_next)
            current_eq_vals.append(val)
            max_violation = max(max_violation, abs(val))

        for g in ineq_constraints:
            val = g(x_next)
            current_ineq_vals.append(val)
            max_violation = max(max_violation, val)

        delta_F = f_current_alm - f_prev
        delta_X = x_next - x_current
        norm_delta_X_sq = np.dot(delta_X, delta_X)

        stop_cond_1 = (delta_F**2) <= epsilon
        stop_cond_2 = norm_delta_X_sq <= epsilon

        if max_violation <= epsilon and (stop_cond_1 or stop_cond_2):
          history_data.append([q,x_next, target_func(x_next), current_eq_vals, current_ineq_vals,lam,beta])
          break

        for i in range(len(lam)):
            lam[i] = lam[i] + 2 * rh[i] * current_eq_vals[i]

        for j in range(len(beta)):
            limit_val = -beta[j] / (2 * rg[j])
            term_max = max(current_ineq_vals[j], limit_val)
            beta[j] = beta[j] + 2 * rg[j] * term_max
        rh = np.minimum(rh * Ch, r_max)
        rg = np.minimum(rg * Cg, r_max)
        f_prev = f_current_alm
        x_current = x_next
        history_data.append([q,x_next, target_func(x_next), current_eq_vals, current_ineq_vals,lam,beta])
    return pd.DataFrame(history_data, columns=['Iter', 'x', 'f(x) Murni', 'h(x)', 'g(x)', 'Lambda', 'Beta'])

In [ ]:
rh_init = [1.0]
rg_init = [1.0]
x0 = [0,0]
eq_constraints=[eq1]
ineq_constraints=[ineq1]
rh_vec=rh_init
rg_vec=rg_init
Ch=3.0
Cg=3.0
Ns=20
r_max = 1e6
epsilon = 1e-6

augmented_lagrange_method(objective_function, x0, eq_constraints, ineq_constraints, Ns, rh_vec, rg_vec, Ch, Cg, r_max, epsilon)

,Iter,x,f(x) Murni,h(x),g(x),Lambda,Beta
0,0,"[0.0, 0.0]",4.000000,[-2.0],[-1.0],[0.0],[0.0]
1,1,"[0.9227556350404019, 1.0391375312907551]",2.957774,[-0.0687152290641393],[0.022724597202487873],[-0.7496880767426841],[0.9994828637036797]
2,2,"[0.9445792685159431, 1.0409609294742965]",2.965138,[-0.02417034879789437],[0.03575724114652212],[-0.7496880767426841],[0.9994828637036797]
3,3,"[0.9712580351457208, 1.0219860129676992]",2.976991,[-0.012202418463259423],[0.019177100734992125],[-0.7496880767426841],[0.9994828637036797]
4,4,"[0.9921942335373684, 1.0060951005062129]",2.992628,[-0.0033232516725876238],[0.0052828627131562556],[-0.7496880767426841],[0.9994828637036797]
5,5,"[0.9991193338323827, 1.0006927443181024]",2.999125,[-0.0003745882314407645],[0.0005993372075874248],[-0.7496880767426841],[0.9994828637036797]
6,6,"[0.9999639396228562, 1.0000283915156214]",2.999964,[-1.5335616615708147e-05],[2.4558014506403936e-05],[-0.7496880767426841],[0.9994828637036797]
7,7,"[0.9999994971498588, 1.0000003915609377]",2.999999,[-2.2257800091907143e-07],[3.359165141603171e-07],[-0.7496880767426841],[0.9994828637036797]
